In [ ]:
import numpy as np
import pandas as pd
import time
import warnings
import lightgbm as lgb
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings('ignore')
t0 = time.time()

INPUT = Path('/kaggle/input/competitions/hbaac-round2')
OUT   = Path('/kaggle/working')

OUT.mkdir(exist_ok=True)

TRAIN_PATH  = INPUT / 'train.csv'
SAMPLE_PATH = INPUT / 'sample_submission.csv'

TOP_N = 1500
TOP_TIER_N = 100         
EVAL_MULTIPLIER = 1.0
SEEDS_FINAL = (42, 123, 7)  


CV_FOLDS = [28, 56, 84]     
STABILITY_STD_PENALTY = 0.35 # select by mean + penalty*std
SOFT_DEADSKU_TOP_ONLY = True # soften dead-SKU rule for top-tier weighted SKUs


LEVEL_CALIBRATION = True
LEVEL_CALIBRATION_CLIP = (0.97, 1.03)
LEVEL_CALIBRATION_STRENGTH = 0.70


print("1. Loading data...")
df = pd.read_csv(TRAIN_PATH, dtype={'Stt': str}, low_memory=False)
df['Date'] = pd.to_datetime(df['Date'])

df['CostAmount'] = pd.to_numeric(df['Cost Amount'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
df['UnitPrice']  = pd.to_numeric(df['UnitPrice'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
df['Profit']     = df['SalesAmount'] - df['CostAmount']

sample = pd.read_csv(SAMPLE_PATH)
skus = (sample['id'].str.replace('_validation', '', regex=False)
                    .str.replace('_evaluation', '', regex=False)
                    .drop_duplicates().tolist())
sku_idx = {s: i for i, s in enumerate(skus)}

date_min = pd.Timestamp('2020-11-17')
date_max = df['Date'].max()
all_dates = pd.date_range(date_min, date_max, freq='D')
dates_idx = pd.DatetimeIndex(all_dates)
date_idx = {d: i for i, d in enumerate(all_dates)}
n_skus, n_days = len(skus), len(all_dates)

daily_qty = df.groupby(['ItemCode', 'Date'], as_index=False)['Quantity'].sum()

df_price = df[df['UnitPrice'] > 0].copy()
daily_price = df_price.groupby(['ItemCode', 'Date'], as_index=False)['UnitPrice'].mean()
daily = daily_qty.merge(daily_price, on=['ItemCode', 'Date'], how='left')

Y = np.zeros((n_skus, n_days), dtype=np.float32)
P = np.zeros((n_skus, n_days), dtype=np.float32)
i_sku = daily['ItemCode'].map(sku_idx).values
i_day = daily['Date'].map(date_idx).values
mask = ~pd.isna(i_sku) & ~pd.isna(i_day)
Y[i_sku[mask].astype(int), i_day[mask].astype(int)] = daily.loc[mask, 'Quantity'].astype(np.float32).values
P[i_sku[mask].astype(int), i_day[mask].astype(int)] = daily.loc[mask, 'UnitPrice'].astype(np.float32).values

P_df = pd.DataFrame(P).replace(0, np.nan)
P = P_df.ffill(axis=1).fillna(0).values

sku_profit = df.groupby('ItemCode')['Profit'].sum().clip(lower=0)
weights = sku_profit.reindex(skus).fillna(0).values
weights = weights / (weights.sum() + 1e-12)

order = np.argsort(-weights)
top_skus = [skus[i] for i in order[:TOP_N]]
top_indices = [sku_idx[s] for s in top_skus]
TOP_TIER_SET = set(top_skus[:TOP_TIER_N])  # ~50% of total weight
print(f"   Top {TOP_TIER_N} SKUs cumulative weight: {weights[order[:TOP_TIER_N]].sum()*100:.1f}%")


tet_dates = pd.to_datetime(['2021-02-12', '2022-02-01', '2023-01-22',
                            '2024-02-10', '2025-01-29', '2026-02-17'])
hung_king_dates = pd.to_datetime(['2021-04-21', '2022-04-10', '2023-04-29',
                                  '2024-04-18', '2025-04-07', '2026-04-26'])


In [ ]:
def build_global_features(Y_mat, P_mat, dates, start_idx, end_idx):
    valid_start = 84
    n_skus_local = Y_mat.shape[0]
    n_time = end_idx - start_idx
    Y_sub = Y_mat[:, start_idx:end_idx]; P_sub = P_mat[:, start_idx:end_idx]
    d_sub = dates[start_idx:end_idx]
    y = np.clip(Y_sub[:, valid_start:].flatten(), 0, None)
    X = pd.DataFrame()
    X['sku_id'] = np.repeat(np.arange(n_skus_local), n_time - valid_start)
    X['sku_id'] = X['sku_id'].astype('category')
    X['lag_7']  = Y_sub[:, valid_start-7 : -7].flatten()
    X['lag_14'] = Y_sub[:, valid_start-14: -14].flatten()
    X['lag_28'] = Y_sub[:, valid_start-28: -28].flatten()
    rmean_28 = np.zeros((n_skus_local, n_time - valid_start), dtype=np.float32)
    for t in range(valid_start, n_time):
        rmean_28[:, t-valid_start] = Y_sub[:, t-28:t].mean(axis=1)
    X['rmean_28'] = rmean_28.flatten()
    X['price'] = P_sub[:, valid_start:].flatten()
    X['price_lag_28'] = P_sub[:, valid_start-28:-28].flatten()
    d_valid = d_sub[valid_start:]
    X['dow']   = np.tile(d_valid.dayofweek.values, n_skus_local)
    X['dom']   = np.tile(d_valid.day.values, n_skus_local)
    X['month'] = np.tile(d_valid.month.values, n_skus_local)
    is_solar = ((d_valid.day==30)&(d_valid.month==4)) | ((d_valid.day==1)&(d_valid.month==5)) | \
               ((d_valid.day==2)&(d_valid.month==9)) | ((d_valid.day==1)&(d_valid.month==1))
    X['is_solar']     = np.tile(is_solar.astype(int), n_skus_local)
    X['is_tet']       = np.tile(d_valid.isin(tet_dates).astype(int), n_skus_local)
    X['is_hung_king'] = np.tile(d_valid.isin(hung_king_dates).astype(int), n_skus_local)
    return X, y


def train_and_predict_global_lgbm(Y_all, P_all, top_idx, dates_full,
                                   train_end_offset, horizon, seeds=(42,)):
    """v10's function + multi-seed averaging (only for final, train_end_offset==0)."""
    if train_end_offset == 0 and len(seeds) > 1:
        use_seeds = seeds
        print(f"  -> Training Global LGBM (FINAL, {len(use_seeds)} seeds)...")
    else:
        use_seeds = (42,)
        print(f"  -> Training Global LGBM (Offset: -{train_end_offset} days, 1 seed)...")

    end_idx = n_days - train_end_offset
    start_idx = max(0, end_idx - 600)
    Y_mat = Y_all[top_idx, :]
    P_mat = P_all[top_idx, :]
    X_train, y_train = build_global_features(Y_mat, P_mat, dates_full, start_idx, end_idx)

    all_preds = []
    for seed in use_seeds:
        params = dict(
            objective='tweedie', tweedie_variance_power=1.1,
            learning_rate=0.05, num_leaves=63, min_data_in_leaf=100,
            feature_fraction=0.8, verbosity=-1, seed=seed, force_row_wise=True,
            device_type='gpu', gpu_platform_id=0, gpu_device_id=0
        )
        try:
            booster = lgb.train(params, lgb.Dataset(X_train, label=y_train,
                                                    categorical_feature=['sku_id']),
                                num_boost_round=400)
        except Exception:
            if seed == use_seeds[0]:
                print("     [Fallback] GPU not available -> CPU...")
            params.pop('device_type', None); params.pop('gpu_platform_id', None); params.pop('gpu_device_id', None)
            booster = lgb.train(params, lgb.Dataset(X_train, label=y_train,
                                                    categorical_feature=['sku_id']),
                                num_boost_round=400)

        preds_matrix = np.zeros((len(top_idx), horizon), dtype=np.float32)
        hist_Y = np.copy(Y_mat[:, :end_idx])
        hist_P = np.copy(P_mat[:, :end_idx])
        for step in range(horizon):
            d_target = dates_full[end_idx - 1] + pd.Timedelta(days=step + 1)
            X_step = pd.DataFrame()
            X_step['sku_id'] = np.arange(len(top_idx))
            X_step['sku_id'] = X_step['sku_id'].astype('category')
            X_step['lag_7']  = hist_Y[:, -7]
            X_step['lag_14'] = hist_Y[:, -14]
            X_step['lag_28'] = hist_Y[:, -28]
            X_step['rmean_28'] = hist_Y[:, -28:].mean(axis=1)
            X_step['price'] = hist_P[:, -1]
            X_step['price_lag_28'] = hist_P[:, -28]
            X_step['dow']   = d_target.dayofweek
            X_step['dom']   = d_target.day
            X_step['month'] = d_target.month
            X_step['is_solar'] = int(
                (d_target.day==30 and d_target.month==4) or
                (d_target.day==1  and d_target.month==5) or
                (d_target.day==2  and d_target.month==9) or
                (d_target.day==1  and d_target.month==1))
            X_step['is_tet']       = int(d_target in tet_dates)
            X_step['is_hung_king'] = int(d_target in hung_king_dates)
            step_preds = np.clip(booster.predict(X_step), 0, None)
            preds_matrix[:, step] = step_preds
            hist_Y = np.hstack([hist_Y, step_preds.reshape(-1, 1)])
            hist_P = np.hstack([hist_P, hist_P[:, -1].reshape(-1, 1)])
        all_preds.append(preds_matrix)
    return np.mean(all_preds, axis=0)


print("\n2. Pre-computing Global LGBM (3 CV folds + final w/ multi-seed)...")
global_cv_preds = {
    off: train_and_predict_global_lgbm(Y, P, top_indices, dates_idx, off, 28)
    for off in CV_FOLDS
}
global_final_preds = train_and_predict_global_lgbm(Y, P, top_indices, dates_idx,
                                                    0, 56, seeds=SEEDS_FINAL)

def baseline_forecast(Y_tr, dates_tr, dow_target, trim_window=84, trim_hi=7,
                     dow_window=84, dow_trim=2, alpha=0.5):
    """v10 baseline (trimmed tail mean + DoW mean blend)."""
    tail = np.clip(Y_tr[:, -trim_window:], 0, None)
    s = np.sort(tail, axis=1)
    if trim_hi > 0: s = s[:, :-trim_hi]
    tmean = s.mean(axis=1, keepdims=True)
    last_idx = np.arange(Y_tr.shape[1]-dow_window, Y_tr.shape[1])
    dow_tr = dates_tr[last_idx].dayofweek.values
    Y_last = np.clip(Y_tr[:, last_idx], 0, None)
    dow_means = np.zeros((Y_tr.shape[0], 7), dtype=np.float32)
    for d in range(7):
        m = dow_tr == d
        if m.sum():
            vals = Y_last[:, m]
            if dow_trim > 0 and vals.shape[1] > dow_trim:
                vals = np.sort(vals, axis=1)[:, :-dow_trim]
            dow_means[:, d] = vals.mean(axis=1)
    pred = alpha * np.tile(tmean, (1, len(dow_target))) + (1-alpha) * dow_means[:, dow_target]
    return np.clip(pred, 0, None)


def fc_ets(y_tr, h, damped=True):
    nz = np.flatnonzero(y_tr > 0)
    if len(nz) == 0: return np.zeros(h)
    y_eff = y_tr[nz[0]:]
    if len(y_eff) < 60: return np.full(h, np.mean(y_eff))
    try:
        return np.clip(ExponentialSmoothing(y_eff, trend='add', damped_trend=damped,
                                            seasonal='add', seasonal_periods=7,
                                            initialization_method="estimated")
                       .fit(optimized=True).forecast(h), 0, None)
    except Exception:
        return np.full(h, np.mean(y_eff[-28:]))


def fc_croston(y_tr, h, alpha=0.1):
    nz = np.flatnonzero(y_tr > 0)
    if len(nz) == 0: return np.zeros(h)
    z_est = y_tr[nz[0]]; p_est = 1.0; q = 1
    for i in range(len(y_tr)):
        if y_tr[i] > 0:
            z_est = alpha * y_tr[i] + (1 - alpha) * z_est
            p_est = alpha * q        + (1 - alpha) * p_est
            q = 1
        else: q += 1
    return np.full(h, np.clip(z_est / p_est if p_est > 0 else 0, 0, None))



def fc_holt(y_tr, h, damped=True):
    """Holt's linear trend (no seasonality) — better for lumpy/trending series."""
    nz = np.flatnonzero(y_tr > 0)
    if len(nz) == 0: return np.zeros(h)
    y_eff = y_tr[nz[0]:]
    if len(y_eff) < 30: return np.full(h, np.mean(y_eff))
    try:
        return np.clip(ExponentialSmoothing(y_eff, trend='add', damped_trend=damped,
                                            seasonal=None,
                                            initialization_method="estimated")
                       .fit(optimized=True).forecast(h), 0, None)
    except Exception:
        return np.full(h, np.mean(y_eff[-28:]))


def fc_lin_drift(y_tr, h, window=90):
    """Linear regression on recent window with conservative cap.

    v13.1 PATCH: linear drift can spike on lumpy SKUs. Cap it using
    recent non-negative demand so it can capture trend without exploding.
    """
    tail = y_tr[-window:] if len(y_tr) >= window else y_tr
    tail_pos = np.clip(tail, 0, None)
    if len(tail) < 14:
        return np.full(h, max(0, float(tail_pos.mean())))
    t = np.arange(len(tail), dtype=np.float64)
    try:
        slope, intercept = np.polyfit(t, tail, 1)
    except Exception:
        return np.full(h, max(0, float(tail_pos.mean())))

    forecast = intercept + slope * np.arange(len(tail), len(tail) + h)
    forecast = np.clip(forecast, 0, None)

    hist_pos = np.clip(y_tr[-180:] if len(y_tr) >= 180 else y_tr, 0, None)
    recent_mean = max(float(tail_pos.mean()), 1e-6)
    cap = max(3.0 * recent_mean, float(np.percentile(hist_pos, 95)) if len(hist_pos) else 0.0)
    forecast = np.minimum(forecast, cap)
    return forecast.astype(np.float32)


def drift_correct(baseline_pred, y_tr, alpha=0.5):
    """Apply drift correction to baseline when recent trend is strong (>15% change)."""
    if len(y_tr) < 56: return baseline_pred
    recent_28 = y_tr[-28:].mean()
    prev_28   = y_tr[-56:-28].mean()
    if prev_28 < 0.5: return baseline_pred
    trend_factor = recent_28 / prev_28
    if abs(trend_factor - 1) < 0.15: return baseline_pred
    # Half-shrink correction (conservative) and clamp to [0.5, 2.0]
    correction = np.clip(1 + alpha * (trend_factor - 1), 0.5, 2.0)
    return np.clip(baseline_pred * correction, 0, None)



def naive_denom(y):
    nz = np.flatnonzero(y != 0)
    if len(nz) == 0: return np.nan
    y_eff = y[nz[0]:]
    d = float((np.diff(y_eff) ** 2).mean())
    return d if d > 1e-12 else np.nan


def rmsse(y, p, d):
    if not d or d <= 0: return np.nan
    return np.sqrt(((y - p) ** 2).mean() / d)


H = 28
folds = CV_FOLDS
best_models = {}


In [ ]:
def get_calib_tier(idx_in_top):
    if idx_in_top < 100:
        return 'top_001_100'
    elif idx_in_top < 500:
        return 'top_101_500'
    elif idx_in_top < TOP_N:
        return 'top_501_1500'
    return None

calib_sums = {
    tier: {off: {'actual': 0.0, 'pred': 0.0} for off in folds}
    for tier in ['top_001_100', 'top_101_500', 'top_501_1500']
}
calib_ratios = {}

print("\n3. Auto-Ensemble CV (with trend-aware methods + per-tier rule)...")
for idx_in_top, sku in enumerate(tqdm(top_skus, desc="CV Hybrid")):
    i = sku_idx[sku]
    y_full = Y[i, :]
    is_top_tier = sku in TOP_TIER_SET

    # 8 methods in pool
    cv_scores = {k: [] for k in ['base', 'base_drift', 'global_lgb',
                                  'ets', 'holt', 'cros', 'lin_drift', 'ens_all']}


    cv_pred_cache = {k: [] for k in cv_scores.keys()}
    cv_actual_cache = []

    for end_off in folds:

        train_cut = n_days - end_off - H
        holdout_start = train_cut
        holdout_end = train_cut + H

        y_tr = y_full[:train_cut]
        y_ho = y_full[holdout_start:holdout_end]
        d_tr = dates_idx[:train_cut]
        d_ho = dates_idx[holdout_start:holdout_end]
        dow_ho = d_ho.dayofweek.values
        denom  = naive_denom(y_tr)

        p_base       = baseline_forecast(y_tr.reshape(1, -1), d_tr, dow_ho)[0]
        p_base_drift = drift_correct(p_base, y_tr)
        p_global_lgb = global_cv_preds[end_off][idx_in_top, :]
        p_ets        = fc_ets(y_tr, H, damped=True)
        p_holt       = fc_holt(y_tr, H, damped=True)
        p_cros       = fc_croston(y_tr, H)
        p_lin_drift  = fc_lin_drift(y_tr, H)
        p_ens_all    = (p_base + p_global_lgb + p_ets + p_holt + p_cros) / 5.0

        preds_dict = {
            'base': p_base, 'base_drift': p_base_drift, 'global_lgb': p_global_lgb,
            'ets': p_ets, 'holt': p_holt, 'cros': p_cros,
            'lin_drift': p_lin_drift, 'ens_all': p_ens_all,
        }

        apply_sun0  = (y_tr[-84:][d_tr[-84:].dayofweek == 6].sum() == 0)
        sun_mask_ho = (dow_ho == 6)

        cv_actual_cache.append(y_ho)
        for model_name, p in preds_dict.items():
            if apply_sun0:
                p = np.where(sun_mask_ho, 0.0, p)
            p = np.clip(p, 0, None)
            cv_pred_cache[model_name].append(p)
            cv_scores[model_name].append(rmsse(y_ho, p, denom))

    mean_scores = {k: np.nanmean(v) for k, v in cv_scores.items()}
    std_scores  = {k: np.nanstd(v)  for k, v in cv_scores.items()}
    penalty_scores = {k: mean_scores[k] + STABILITY_STD_PENALTY * std_scores[k] for k in mean_scores}

    best_algo = min(penalty_scores, key=lambda k: penalty_scores[k] if not np.isnan(penalty_scores[k]) else np.inf)
    base_score = penalty_scores['base']

    if not is_top_tier:
        # Non-top SKUs: keep v10 safety net (avoids picking noisy algo by accident)
        if penalty_scores[best_algo] > base_score * 0.99:
            best_algo = 'base'
    # else: top 100 SKUs — trust best_algo (no fallback to base)

    best_models[sku] = {'best_algo': best_algo,
                        'cv_mean': mean_scores[best_algo],
                        'cv_std':  std_scores[best_algo],
                        'is_top': is_top_tier}


    tier = get_calib_tier(idx_in_top)
    if LEVEL_CALIBRATION and tier is not None:
        for fold_pos, end_off in enumerate(folds):
            calib_sums[tier][end_off]['actual'] += float(np.sum(cv_actual_cache[fold_pos]))
            calib_sums[tier][end_off]['pred'] += float(np.sum(cv_pred_cache[best_algo][fold_pos]))

if LEVEL_CALIBRATION:
    print("\n=== v13.2 Level Calibration Ratios ===")
    lo, hi = LEVEL_CALIBRATION_CLIP
    for tier, fold_sums in calib_sums.items():
        fold_ratios = []
        for end_off in folds:
            actual_sum = fold_sums[end_off]['actual']
            pred_sum = fold_sums[end_off]['pred']
            if pred_sum > 1e-9 and np.isfinite(actual_sum) and np.isfinite(pred_sum):
                fold_ratios.append(actual_sum / pred_sum)
        if len(fold_ratios) == 0:
            raw_ratio = 1.0
        else:
            raw_ratio = float(np.nanmedian(fold_ratios))
        clipped_ratio = float(np.clip(raw_ratio, lo, hi))
        # Shrink toward 1.0 so this remains a CV-guided calibration, not a magic multiplier.
        final_ratio = 1.0 + LEVEL_CALIBRATION_STRENGTH * (clipped_ratio - 1.0)
        calib_ratios[tier] = final_ratio
        print(f"  {tier}: raw={raw_ratio:.5f}, clipped={clipped_ratio:.5f}, applied={final_ratio:.5f}, folds={np.round(fold_ratios, 5).tolist()}")
else:
    calib_ratios = {'top_001_100': 1.0, 'top_101_500': 1.0, 'top_501_1500': 1.0}

print("\n=== CV Summary ===")
print(f"Overall: {dict(Counter(best_models[s]['best_algo'] for s in top_skus))}")
print(f"Top {TOP_TIER_N}: {dict(Counter(best_models[s]['best_algo'] for s in top_skus[:TOP_TIER_N]))}")


print("\n4. Generating final forecast...")
forecast_dates = pd.date_range(date_max + pd.Timedelta(days=1), periods=56, freq='D')
dow_fc = forecast_dates.dayofweek.values
pred_56 = baseline_forecast(Y, dates_idx, dow_fc)

for idx_in_top, sku in enumerate(tqdm(top_skus, desc="Overriding Top SKUs")):
    i = sku_idx[sku]
    y_full = Y[i, :]
    algo = best_models[sku]['best_algo']

    p_base       = pred_56[i, :].copy()
    p_base_drift = drift_correct(p_base, y_full)
    p_global_lgb = global_final_preds[idx_in_top, :]
    p_ets        = fc_ets(y_full, 56, damped=True)
    p_holt       = fc_holt(y_full, 56, damped=True)
    p_cros       = fc_croston(y_full, 56)
    p_lin_drift  = fc_lin_drift(y_full, 56)

    if   algo == 'base':       final_p = p_base
    elif algo == 'base_drift': final_p = p_base_drift
    elif algo == 'global_lgb': final_p = p_global_lgb
    elif algo == 'ets':        final_p = p_ets
    elif algo == 'holt':       final_p = p_holt
    elif algo == 'cros':       final_p = p_cros
    elif algo == 'lin_drift':  final_p = p_lin_drift
    elif algo == 'ens_all':    final_p = (p_base + p_global_lgb + p_ets + p_holt + p_cros) / 5.0
    else:                      final_p = p_base

    # Sunday=0 heuristic (v10)
    if (y_full[-84:][dates_idx[-84:].dayofweek == 6].sum() == 0):
        final_p = np.where(dow_fc == 6, 0.0, final_p)

 
    if SOFT_DEADSKU_TOP_ONLY and sku in TOP_TIER_SET:
        if y_full[-180:].sum() <= 0:
            final_p = np.zeros(56)
        elif y_full[-84:].sum() <= 0:
            final_p *= 0.15
        elif y_full[-56:].sum() <= 0:
            final_p *= 0.35
    else:
        if y_full[-28:].sum() <= 0:
            final_p = np.zeros(56)

    pred_56[i, :] = np.clip(final_p, 0, None)


if LEVEL_CALIBRATION:
    print("\n4b. Applying CV-derived tier-level calibration...")
    for idx_in_top, sku in enumerate(top_skus):
        tier = get_calib_tier(idx_in_top)
        if tier is not None:
            i = sku_idx[sku]
            pred_56[i, :] *= calib_ratios.get(tier, 1.0)
    pred_56 = np.clip(pred_56, 0, None)

print("\n5. Exporting submission...")
val_preds  = pred_56[:, :28]
eval_preds = pred_56[:, 28:] * EVAL_MULTIPLIER

F_cols = [f'F{i+1}' for i in range(28)]
val = pd.DataFrame(val_preds,  columns=F_cols)
val.insert(0, 'id', [f'{s}_validation' for s in skus])
ev = pd.DataFrame(eval_preds, columns=F_cols)
ev.insert(0, 'id', [f'{s}_evaluation' for s in skus])

sub = pd.concat([val, ev], ignore_index=True)[sample.columns]
out_csv_path = OUT / "submission_v13_2_level_calibration.csv"
sub.to_csv(out_csv_path, index=False)

print(f"\n✅ Saved: {out_csv_path}")
print(f"Total time: {time.time()-t0:.1f}s")